# 4. The Conditional Chain (RunnableBranch)

A **conditional chain** picks **one** path out of several based on a condition — like `if / elif /
else` for chains. Route a math question to a math chain, a coding question to a code chain, everything
else to a general chain.

---

## 1. Simple Definition

> **Kid version:** It's a **choose-your-path** slide at the playground 🛝. At the top there's a sign:
> "Wet? Go left. Dry? Go right." Depending on the condition, you go down **only one** slide, not all
> of them. A conditional chain reads the input and sends it down just one branch.

**Professional definition:** A `RunnableBranch` evaluates a list of `(condition, runnable)` pairs in
order and runs the **first** branch whose condition returns `True`; if none match, it runs a **default**
branch. It implements if/elif/else routing over Runnables.

```python
from langchain_core.runnables import RunnableBranch

chain = RunnableBranch(
    (lambda x: x["type"] == "math", math_chain),      # if math   → math_chain
    (lambda x: x["type"] == "code", code_chain),      # elif code → code_chain
    general_chain,                                     # else      → general_chain
)
```

---

## 2. Why Does It Exist?

**The problem:** Not every input should be handled the same way. A customer-support bot should route a
**refund** question, a **technical** question, and a **sales** question to *different* specialized
prompts/chains. One-size-fits-all prompts perform worse.

### Before (manual if/else around invoke calls)

```python
if is_math(query):
    result = math_chain.invoke(query)
elif is_code(query):
    result = code_chain.invoke(query)
else:
    result = general_chain.invoke(query)
# Works, but it's not a Runnable — can't compose, stream, batch, or nest it in a bigger chain.
```

### After (a conditional chain — still a Runnable)

```python
chain = RunnableBranch(
    (is_math, math_chain),
    (is_code, code_chain),
    general_chain,
)
chain.invoke(query)     # picks one branch, and the whole thing composes/streams/batches
```

The key gain: the routing logic becomes **a Runnable itself**, so it plugs into bigger chains and
keeps streaming/batching/async.

**Where you'll use it:** intent routing (support topics), difficulty routing (easy→small model,
hard→big model), language routing, "is this answerable? if not, ask for clarification," guardrails.

---

## 3. Real-Life Analogy

**A telephone helpdesk menu** ☎️. "Press 1 for billing, 2 for technical support, 3 for everything
else." Your choice sends you to **one** department. The menu (router) inspects your need and connects
you to the right specialist — exactly what a conditional chain does.

Other analogies: a **train switch/junction** (one track chosen), a **triage nurse** (routes patients
by symptom), a **mail sorter**.

---

## 4. Where It Fits in LangChain Architecture

```
                    ┌── condition 1 True? ──► branch_1
   input ──► router ┤── condition 2 True? ──► branch_2
                    └── else ──────────────► default_branch
                                 │
                                 ▼
                        output of the chosen branch only
```

Contrast with parallel: **parallel runs ALL branches**; **conditional runs ONE**. Both take a
set of branches, but the behavior is opposite.

---

## 5. Internal Working — how a branch is chosen

```
  chain.invoke({"type": "code", "q": "reverse a list in python"})
        │
        ▼
  evaluate conditions IN ORDER:
     cond1: x["type"] == "math"  → False   ✗
     cond2: x["type"] == "code"  → True    ✓  ← STOP here
        │
        ▼
  run ONLY code_chain on the input
        │
        ▼
  return code_chain's output
  (if no condition had matched → run the default branch)
```

Rules to remember:
- Conditions are checked **top to bottom**; the **first** `True` wins (order matters, like `elif`).
- The **last argument is the default** (the `else`) — it has **no condition** and is required.
- Only **one** branch runs; the input is passed to that branch as-is.

---

## 6. The two ways to route

### Way A — RunnableBranch (explicit if/elif/else)

**Definition:** A list of `(condition, runnable)` pairs plus a default.

**Why it exists:** Clear, declarative multi-way branching.

**When developers use it:** When you have a few well-defined conditions.

```python
from langchain_core.runnables import RunnableBranch

branch = RunnableBranch(
    (lambda x: len(x["text"]) > 1000, summarize_then_answer),
    (lambda x: "?" in x["text"],       direct_answer),
    fallback_chain,                    # default
)
```

---

### Way B — RunnableLambda that returns a chain (custom routing)

**Definition:** A function inspects the input and **returns** the Runnable to use; LangChain runs it.

**Why it exists:** Maximum flexibility — arbitrary Python logic (or an LLM classifier) chooses the path.

**When developers use it:** Complex or dynamic routing, or routing based on an LLM's classification.

```python
from langchain_core.runnables import RunnableLambda

def route(info):
    if info["topic"] == "billing":
        return billing_chain
    elif info["topic"] == "tech":
        return tech_chain
    return general_chain

full = RunnableLambda(route)     # returning a Runnable → it gets invoked on the input
```

> Returning a Runnable from a `RunnableLambda` is a powerful idiom: the function *decides*, LangChain
> *executes* the chosen chain.

---

## 7. The classic pattern: classify, then branch

Routing usually has two stages: (1) an LLM (or rule) **classifies** the input, (2) a branch **routes**
on that classification.

```python
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableBranch, RunnablePassthrough

# Stage 1: classify the topic into one word
classifier = (
    ChatPromptTemplate.from_template(
        "Classify the question as 'math', 'code', or 'other'. "
        "Answer with one word only.\n\nQuestion: {input}"
    )
    | model | StrOutputParser()
)

# Sub-chains for each topic
math_chain  = ChatPromptTemplate.from_template("Solve step by step:\n{input}") | model | StrOutputParser()
code_chain  = ChatPromptTemplate.from_template("Write code for:\n{input}")     | model | StrOutputParser()
other_chain = ChatPromptTemplate.from_template("Answer:\n{input}")             | model | StrOutputParser()

# Stage 2: branch on the classification
branch = RunnableBranch(
    (lambda x: x["topic"] == "math", math_chain),
    (lambda x: x["topic"] == "code", code_chain),
    other_chain,   # default
)

# Wire together: keep the original input, add a "topic" field, then branch
full_chain = RunnablePassthrough.assign(topic=classifier) | branch

print(full_chain.invoke({"input": "How do I reverse a list in Python?"}))
# classifier → "code"  → routes to code_chain → returns code
```

Here `RunnablePassthrough.assign(topic=classifier)` adds the classifier's result as `topic`
while keeping the original `input`, so the branch conditions can read `x["topic"]` and the sub-chains
can still read `x["input"]`.

---

## 8. Conditions: what can they be?

- A **lambda/function** taking the input and returning a bool: `lambda x: x["n"] > 10`.
- Any Python predicate — string checks, regex, length, presence of a key, etc.
- The result of an **earlier classification** step stored on the input (the pattern above).

Each condition receives the **whole input** passed to the branch, so make sure the field it checks is
present (that's why we `assign` the `topic` first).

In [1]:
from langchain_core.runnables import RunnableBranch

# Branch 1
large_number = lambda x: f"{x} is a large number"

# Default branch
small_number = lambda x: f"{x} is a small number"

# Conditional chain
chain = RunnableBranch((lambda x: x > 10, large_number), small_number)

# Test
print(chain.invoke(20))
print(chain.invoke(5))

20 is a large number
5 is a small number


In [3]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableBranch, RunnableLambda
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from pydantic import BaseModel, Field
from typing import Literal

In [4]:
# step1: Initialize the language model
llm = ChatOllama(model="qwen3:8b")

In [5]:
# step2: Initialize output parser
parser = StrOutputParser()

class Feedback(BaseModel):
    sentiment: Literal['positive', 'negative'] = Field(description='Give the sentiment of the feedback')

parser2 = PydanticOutputParser(pydantic_object=Feedback)

In [6]:
# step3: Create the prompt templates
classify_sentiment_prompt = PromptTemplate(template='Classify the sentiment of the following feedback text into postive or negative \n {feedback} \n {format_instruction}',
                                           input_variables=['feedback'],
                                           partial_variables={'format_instruction':parser2.get_format_instructions()})

positive_feedback_prompt = PromptTemplate(template='Write an appropriate response to this positive feedback \n {feedback}',
                                          input_variables=['feedback'])

negative_feedback_prompt = PromptTemplate(template='Write an appropriate response to this negative feedback \n {feedback}',
                                          input_variables=['feedback'])

In [7]:
classifier_chain = classify_sentiment_prompt | llm | parser2

branch_chain = RunnableBranch(
    (lambda x:x.sentiment == 'positive', positive_feedback_prompt | llm | parser),
    (lambda x:x.sentiment == 'negative', negative_feedback_prompt | llm | parser),
    RunnableLambda(lambda x: "could not find sentiment")
)

chain = classifier_chain | branch_chain

In [8]:
result = chain.invoke({'feedback': 'This is a beautiful phone'})

In [9]:
print(result)

"Thank you so much for your kind words! I'm so glad you enjoyed it. Your support means a lot, and I truly value your feedback. If you ever need anything else, feel free to reach out—I'm always here to help! 😊"


In [10]:
chain.get_graph().print_ascii()

    +-------------+      
    | PromptInput |      
    +-------------+      
            *            
            *            
            *            
   +----------------+    
   | PromptTemplate |    
   +----------------+    
            *            
            *            
            *            
     +------------+      
     | ChatOllama |      
     +------------+      
            *            
            *            
            *            
+----------------------+ 
| PydanticOutputParser | 
+----------------------+ 
            *            
            *            
            *            
       +--------+        
       | Branch |        
       +--------+        
            *            
            *            
            *            
    +--------------+     
    | BranchOutput |     
    +--------------+     
